# Vulnerability 4 — Unsafe Data Pipeline Configuration

This notebook demonstrates an **unsafe data loading function** that:
- Performs no validation
- Accepts any CSV
- Cannot detect poisoning, schema changes, or oversized files

This is a classic AI‑specific vulnerability: **data poisoning and integrity issues**.


## 1. Vulnerable Data Loader

A typical ML helper function:


In [ ]:
import os
import pandas as pd
import numpy as np

print("="*70)
print("VULNERABILITY 4: UNSAFE DATA PIPELINE")
print("="*70)

def load_training_data_unsafe(file_path):
    """Loads training data without any validation.
    
    ❌ No file size check
    ❌ No schema validation
    ❌ No integrity verification
    """
    data = pd.read_csv(file_path)
    return data

print("\n[Vulnerable Code Pattern]\n")
print("def load_training_data_unsafe(file_path):\n    # No file size check\n    # No schema validation\n    # No integrity verification\n    data = pd.read_csv(file_path)\n    return data")

## 2. Creating Clean and Poisoned Data

We simulate a clean dataset and a poisoned version with **flipped labels**.


In [ ]:
print("\n[Creating Poisoned Data]")

np.random.seed(42)
clean_data = pd.DataFrame({
    'feature1': np.random.randn(100),
    'feature2': np.random.randn(100),
    'label': np.random.randint(0, 2, 100)
})
clean_data.to_csv('clean_data.csv', index=False)

poisoned_data = clean_data.copy()
poisoned_indices = np.random.choice(100, 10, replace=False)
poisoned_data.loc[poisoned_indices, 'label'] = 1 - poisoned_data.loc[poisoned_indices, 'label']
poisoned_data.to_csv('poisoned_data.csv', index=False)

print("✓ Created clean_data.csv (legitimate)")
print("✓ Created poisoned_data.csv (10% labels flipped)")

## 3. Loading Data Without Validation

The vulnerable loader treats both files as equally valid.


In [ ]:
print("\n[Loading Data Without Validation]")
clean = load_training_data_unsafe('clean_data.csv')
poisoned = load_training_data_unsafe('poisoned_data.csv')

print(f"Clean data loaded: {len(clean)} rows")
print(f"Poisoned data loaded: {len(poisoned)} rows")
print("\n⚠️  No validation performed — poisoned data accepted!")

# Cleanup
os.remove('clean_data.csv')
os.remove('poisoned_data.csv')

## 4. Impact, Detection, Prevention

### Impact
- Data poisoning attacks go undetected
- Denial of service via huge files
- Schema manipulation attacks
- No integrity verification of data sources

### Detection
- Static analysis can flag missing validation checks
- Semgrep/Bandit rules can look for unvalidated file reads

### Prevention
- Validate schemas (columns, types, ranges)
- Enforce file size limits
- Verify checksums or signatures
- Restrict data sources to trusted locations
